# 🧠 类人脑双系统 AI - Telegram Bot 一键部署

基于 Qwen2.5-0.5B 模型，集成海马体记忆系统和 STDP 学习机制。

## 功能特性
- 💭 自发内心独白流
- 🧠 海马体记忆系统
- ⚡ STDP 在线学习
- 📡 Telegram Bot 接口

## 1️⃣ 配置参数

请填写您的 Telegram Bot Token（从 @BotFather 获取）

In [ ]:
# @title 配置 Telegram Bot Token
# @markdown 请输入您的 Telegram Bot Token

TELEGRAM_BOT_TOKEN = ""  # @param {type:"string"}

# 验证 Token 格式
if not TELEGRAM_BOT_TOKEN:
    print("⚠️ 请填写 TELEGRAM_BOT_TOKEN")
    print("获取方式：在 Telegram 中搜索 @BotFather，发送 /newbot 创建机器人")
elif len(TELEGRAM_BOT_TOKEN.split(':')) != 2:
    print("⚠️ Token 格式错误，正确格式：123456789:ABCdefGHIjklMNOpqrsTUVwxyz")
else:
    print(f"✅ Token 格式正确")

## 2️⃣ 安装依赖

In [ ]:
# @title 安装系统依赖
!apt-get update -qq
!apt-get install -y -qq git git-lfs
!git lfs install
print("✅ 系统依赖安装完成")

In [ ]:
# @title 克隆项目代码
import os

# 删除旧目录（如果存在）
!rm -rf stdpbrain

# 克隆项目
!git clone --branch glm https://github.com/ctz168/stdpbrain.git

# 切换到项目目录
os.chdir('/content/stdpbrain')
print(f"✅ 项目克隆完成，当前目录: {os.getcwd()}")

In [ ]:
# @title 安装 Python 依赖
!pip install -q torch transformers accelerate sentencepiece
!pip install -q python-telegram-bot httpx apscheduler
!pip install -q numpy scipy
print("✅ Python 依赖安装完成")

## 3️⃣ 下载模型

In [ ]:
# @title 下载 Qwen2.5-0.5B 模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

model_name = "Qwen/Qwen2.5-0.5B"
model_path = "./models/Qwen3.5-0.8B-Base"

os.makedirs(model_path, exist_ok=True)

print(f"📥 正在下载模型: {model_name}")
print("这可能需要几分钟...")

# 下载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.save_pretrained(model_path)
print("✅ Tokenizer 下载完成")

# 下载模型
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype="auto"
)
model.save_pretrained(model_path)
print("✅ 模型下载完成")

## 4️⃣ 配置并运行 Bot

In [ ]:
# @title 创建配置文件
%%writefile /content/stdpbrain/.env
TELEGRAM_BOT_TOKEN=YOUR_TOKEN_HERE


In [ ]:
# @title 创建启动脚本
%%writefile /content/stdpbrain/run_colab.py
"""
Colab 一键启动脚本
"""
import os
import sys
import asyncio
import logging

# 配置日志
logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

# 添加项目路径
sys.path.insert(0, '/content/stdpbrain')

def main():
    # 导入模块
    from configs.arch_config import default_config
    from core.interfaces import BrainAIInterface
    from telegram_bot.bot import BrainAIBot
    
    # 获取 Token
    token = os.environ.get('TELEGRAM_BOT_TOKEN', '')
    if not token:
        print("❌ 错误：未设置 TELEGRAM_BOT_TOKEN 环境变量")
        print("请在 Colab 中设置：")
        print("import os")
        print("os.environ['TELEGRAM_BOT_TOKEN'] = 'your_token_here'")
        return
    
    print("=" * 60)
    print("🧠 类人脑双系统 AI - Telegram Bot")
    print("=" * 60)
    print()
    
    # 初始化 AI 接口
    print("[1/2] 正在加载 AI 模型...")
    try:
        ai = BrainAIInterface(default_config, device='cuda')
        print("✅ AI 模型加载完成")
    except Exception as e:
        print(f"❌ AI 模型加载失败: {e}")
        return
    
    # 初始化 Bot
    print("[2/2] 正在启动 Telegram Bot...")
    try:
        bot = BrainAIBot(token=token, ai_interface=ai)
        print("✅ Bot 启动成功")
        print()
        print("=" * 60)
        print("🎉 Bot 已启动！在 Telegram 中与您的机器人对话吧！")
        print("=" * 60)
        print()
        
        # 运行 Bot
        bot.run()
        
    except Exception as e:
        print(f"❌ Bot 启动失败: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


In [ ]:
# @title 设置环境变量并启动 Bot
import os

# 设置 Token
if TELEGRAM_BOT_TOKEN:
    os.environ['TELEGRAM_BOT_TOKEN'] = TELEGRAM_BOT_TOKEN
    print(f"✅ Token 已设置")
else:
    print("⚠️ 请先在上方配置 TELEGRAM_BOT_TOKEN")
    print("然后重新运行此单元格")

In [ ]:
# @title 运行 Bot
# @markdown 点击运行后，Bot 将持续运行直到单元格被中断

import os
import sys

# 切换到项目目录
os.chdir('/content/stdpbrain')
sys.path.insert(0, '/content/stdpbrain')

# 检查 Token
if not os.environ.get('TELEGRAM_BOT_TOKEN'):
    print("❌ 错误：未设置 TELEGRAM_BOT_TOKEN")
    print("请先运行上一个单元格设置 Token")
else:
    # 运行 Bot
    !python run_colab.py

## 📝 使用说明

1. **获取 Bot Token**：在 Telegram 中搜索 @BotFather，发送 `/newbot` 创建机器人，获取 Token
2. **填写 Token**：在上方 "配置 Telegram Bot Token" 单元格中填入您的 Token
3. **依次运行**：从上到下依次运行所有单元格
4. **开始对话**：在 Telegram 中找到您的机器人，发送消息即可开始对话

## ⚠️ 注意事项

- Colab 免费版 GPU 有限制（约 12 小时），超时后需要重新运行
- 如果遇到内存不足，可以尝试重启运行时（Runtime -> Restart runtime）
- 模型首次加载需要下载，请耐心等待